# 018 — Blocked diagnostics for the fixed broadcast-aware boosting configuration

Пять отдельных моделей: каждый раз одна публичная минута исключена из обучения вместе с 20 с защитного отступа с обеих сторон. Обучение на остальных публичных минутах (в том числе будущих), оценка только на исключённой. Признаки 016, конфигурация boost_shallow фиксирована до проверки; пороги .5/.7/.9 фиксированы. Родительские слабые модели не обучены на публичных метках.

Это офлайн-перекрёстная диагностика, а не causal-тест. Конфигурация ранее выбрана с использованием публичного префикса, весь отрезок многократно исследован: результат условный и не является независимой или полностью nested оценкой. Соседние кадры и остановки зависимы. Не путать агрегированные out-of-fold прогнозы пяти разных моделей с одной финальной моделью 016.

Run All после 016. Используется подготовка признаков из его ноутбука без выполнения обучающей ячейки. Прогноз каждой исключённой минуты использует полную последовательность входных признаков и офлайн-фильтр; исключённые метки не входят в fit. Все результаты и fold-модели остаются локальными; публикуется только код/конфигурация. Полного конкурсного прогноза H/A/D здесь нет.


In [ ]:
from pathlib import Path
import json
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'games.py').exists())
EXPERIMENT='018_dead_blocked_diagnostics'
parent=json.loads((ROOT/'sol/016_dead_broadcast_features/experiment.ipynb').read_text())
exec(''.join(next(c for c in parent['cells'] if c.get('id')=='phase-0')['source']))
CONFIG=dict(name='boost_shallow',kind='boost',depth=2,leaves=4,leaf=15,trees=150,l2=5.)
CUTOFFS=[.5,.7,.9]
oof_prob=np.full(N,np.nan,np.float32);oof_pred={t:np.zeros(N,bool) for t in CUTOFFS};rows=[]
for fold in tqdm(range(5),desc='Leave one minute out / Исключаем минуту'):
 a=START+fold*1500;b=a+1500
 # No held-out labels or labels within 20 seconds of the held-out block enter fitting.
 # Метки исключённой минуты и защитного окна 20 с не попадают в обучение.
 eligible=public&~((positions>=a-GUARD)&(positions<b+GUARD))
 ids=np.flatnonzero(eligible)[::5];y=truth[ids].astype(int);w=np.ones(len(ids),float)
 assert not ((ids>=a-GUARD)&(ids<b+GUARD)).any()
 for label in [0,1]:
  assert (y==label).any();w[y==label]/=w[y==label].sum()
 w*=len(w)/w.sum()
 model=fit_variant(CONFIG,ids,y,w,f'fold {fold+1}')
 p=predict(model,np.arange(N));oof_prob[a:b]=p[a:b]
 with (OUT/f'fold_{fold+1}_model.pkl').open('wb') as f:pickle.dump(model,f)
 for threshold in CUTOFFS:
  pred=np.zeros(N,bool)
  for _,lo,hi in BLOCKS:pred[lo:hi+1]=filter_segment(p[lo:hi+1]>=threshold,25)
  oof_pred[threshold][a:b]=pred[a:b]
  rows.append(dict(fold=fold+1,threshold=threshold,training_samples=len(ids),**score(truth[a:b],pred[a:b])))
summary={str(t):score(truth[public],oof_pred[t][public]) for t in CUTOFFS}
assert np.isfinite(oof_prob[public]).all()
np.save(OUT/'oof_p_D.npy',oof_prob)
for t in CUTOFFS:
 labels=np.where(public,np.where(oof_pred[t],'D','LIVE'),'X')
 pd.DataFrame({'frame':np.arange(N),'label':labels}).to_csv(OUT/f'oof_{t}_predictions.csv',index=False)
pd.DataFrame(rows).to_csv(OUT/'fold_metrics.csv',index=False)
save_json(OUT/'results.json',dict(configuration=CONFIG,thresholds=CUTOFFS,fold_metrics=rows,pooled_oof=summary,
 context='five fitted models; public labels outside held minute plus 20s guard; configuration selected earlier on public data; offline inputs; not independent'))
save_json(OUT/'configuration.json',dict(model=CONFIG,thresholds=CUTOFFS,guard_frames=GUARD,public_range=[START,END],min_dead_frames=25,max_live_gap_frames=10))
progress(stage='completed',fits=5)
print(json.dumps(summary,indent=2))
